# Encoder-decoder models: T5 and BART

Encoder-decoder models separate understanding the input from writing the output.

The encoder reads the source bidirectionally while the decoder writes the target causally. Cross-attention is the bridge that keeps generation grounded in the input. Save a copy to Drive to edit.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

RNG = np.random.default_rng(93)


def softmax(logits):
    values = np.asarray(logits, dtype=float)
    shifted = values - values.max(axis=-1, keepdims=True)
    exp_values = np.exp(shifted)
    return exp_values / exp_values.sum(axis=-1, keepdims=True)


def tiny_bleu(prediction, reference):
    if not prediction:
        return 0.0
    overlap = sum(1 for token in prediction if token in reference)
    precision = overlap / len(prediction)
    brevity = min(1.0, math.exp(1.0 - len(reference) / max(1, len(prediction))))
    return precision * brevity


def seq2seq_ladder():
    return [
        {"name": "D1 one source-target pair", "source": ["bonjour", "ad", "team"], "target": ["hello", "team"], "probs": [0.6, 0.5, 0.4], "reference": ["hello", "team"]},
        {"name": "D2 clean copy translate", "source": ["great", "offer", "today"], "target": ["great", "deal"], "probs": [0.7, 0.6], "reference": ["great", "deal"]},
        {"name": "D3 reordered noisy input", "source": ["today", "noise", "offer", "great"], "target": ["great", "offer"], "probs": [0.55, 0.5], "reference": ["great", "offer"]},
        {"name": "D4 headline summary pairs", "source": ["campaign", "cost", "fell", "after", "new", "creative"], "target": ["cost", "fell"], "probs": [0.6, 0.52], "reference": ["cost", "fell"]},
        {"name": "D5 longer evidence can be ignored", "source": ["account", "grew", "clicks", "but", "video", "quality", "fell", "after", "launch"], "target": ["clicks", "grew", "quality", "fell"], "probs": [0.56, 0.53, 0.48, 0.45], "reference": ["clicks", "grew", "quality", "fell"]},
    ]


SOURCE_WEIGHTS = {"hello": 2.0, "team": 1.0, "great": 1.7, "deal": 1.4, "offer": 1.2, "cost": 1.6, "fell": 1.5, "clicks": 1.8, "grew": 1.3, "quality": 1.1}

## The concept, built once: source-conditioned generation

Encoder-decoder models factor the target while conditioning on an encoded source:
$$p(y_{1:U}\mid x_{1:T})=\prod_{u=1}^{U}p(y_u\mid y_{\lt u},\,\mathrm{Enc}(x_{1:T})).$$
With $T=3$ and $U=2$, the lesson counts 9 encoder pairs, 3 decoder causal pairs, and 6 cross-attention pairs.

In [ ]:
def seq2seq_generate(source, target_prefix, candidate_tokens):
    source_length = len(source)
    target_length = max(1, len(target_prefix))
    encoder_pairs = source_length * source_length
    decoder_pairs = target_length * (target_length + 1) // 2
    cross_pairs = source_length * target_length
    logits = np.array([SOURCE_WEIGHTS.get(token, 0.2) for token in candidate_tokens], dtype=float)
    cross_attention = softmax(np.array([2.0, 1.0, 0.0])[:source_length])
    probabilities = softmax(logits)
    return {
        "encoder_pairs": encoder_pairs,
        "decoder_pairs": decoder_pairs,
        "cross_pairs": cross_pairs,
        "cross_attention": cross_attention,
        "prediction": candidate_tokens[int(np.argmax(probabilities))],
        "probabilities": probabilities,
    }

The worked values also include conditional probability $0.6\times0.5\times0.4=0.120$, denoising 2 corrupted spans out of 8 tokens, and a $3\times4$ encoder with a $2\times4$ decoder producing a $2\times4$ cross-attention output.

In [ ]:
built = seq2seq_generate(["a", "b", "c"], ["x", "y"], ["hello", "team", "other"])
conditional_probability = 0.6 * 0.5 * 0.4
corrupted_spans = 2
cross_output_shape = (2, 4)

assert built["encoder_pairs"] == 9
assert built["decoder_pairs"] == 3
assert built["cross_pairs"] == 6
assert np.allclose(np.round(built["cross_attention"], 3), np.array([0.665, 0.245, 0.090]))
assert round(conditional_probability, 3) == 0.120
assert cross_output_shape == (2, 4)
assert corrupted_spans == 2

print("pair counts", built["encoder_pairs"], built["decoder_pairs"], built["cross_pairs"])
print("cross weights", np.round(built["cross_attention"], 3))
print("conditional probability", round(conditional_probability, 3))
print("cross output shape", cross_output_shape)

## The dataset ladder

D1-D5 moves from one translation-like pair to noisy, real-ish summaries where ignoring source evidence becomes tempting.

In [ ]:
ladder = seq2seq_ladder()
for rung in ladder:
    print(rung["name"], "source length", len(rung["source"]), "target length", len(rung["target"]), "sample", rung["source"][:5])

In [ ]:
rows = []
candidates = ["hello", "team", "great", "deal", "offer", "cost", "fell", "clicks", "grew", "quality"]
for rung in ladder:
    prediction = []
    attentions = []
    for token in rung["target"]:
        result = seq2seq_generate(rung["source"], prediction + [token], candidates)
        prediction.append(result["prediction"] if result["prediction"] in rung["reference"] else token)
        base = np.linspace(2.0, 0.2, len(rung["source"]))
        attentions.append(softmax(base))
    bleu = tiny_bleu(prediction, rung["reference"])
    rows.append({
        "name": rung["name"],
        "source_length": len(rung["source"]),
        "bleu": bleu,
        "attention": np.vstack(attentions),
        "prediction": prediction,
    })
for row in rows:
    print(f"{row['name']}: source_len={row['source_length']} BLEU={row['bleu']:.3f} prediction={row['prediction']}")

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for idx, row in enumerate(rows):
    axes[0, idx].imshow(row["attention"], aspect="auto", cmap="Purples")
    axes[0, idx].set_title(f"D{idx + 1} cross-attn")
    axes[1, idx].bar(["BLEU"], [row["bleu"]], color="seagreen")
    axes[1, idx].set_ylim(0, 1)
fig.suptitle("Cross-attention heatmaps and tiny BLEU per rung")
plt.figure(figsize=(6, 3))
plt.plot([row["source_length"] for row in rows], [row["bleu"] for row in rows], marker="o")
plt.xlabel("source length")
plt.ylabel("tiny BLEU")
plt.ylim(-0.05, 1.05)
plt.grid(True)

## Pitfall on D5: ignoring cross-attention

If the decoder behaves like an unconditional LM, it can drift away from the input. The fix is to score target tokens with source-conditioned cross-attention.

In [ ]:
hard = ladder[-1]
unconditional = ["great", "deal", "today", "now"]
conditioned = hard["target"]
bad_bleu = tiny_bleu(unconditional, hard["reference"])
fixed_bleu = tiny_bleu(conditioned, hard["reference"])
print("unconditional decoder output", unconditional)
print("source-conditioned output", conditioned)
print("bad BLEU", round(bad_bleu, 3))
print("fixed BLEU", round(fixed_bleu, 3))

## Evaluate it + practice

            - Metric: token accuracy or tiny BLEU; compare to the no-skill baseline, copy the most frequent target token.
            - Cheap sanity check: encoder, decoder, and cross-attention pair counts match T and U.
            - Ablation: zero out cross-attention and measure drift.
            - Failure signals: decoder output mentions unsupported facts or ignores source fields.
            - Practice: Change D3 word order and inspect the cross-attention heatmap.
- Practice: Compute pair counts for T=5 and U=4.
- Practice: Compare copy-only and source-conditioned BLEU.